# BrainSegNet — Quick Start Verification (5-Patient Test)
## Run this notebook FIRST before the full training notebook

**What this does:** Runs a 5-patient / 3-epoch mini version of the full pipeline  
**Purpose:** Confirm everything works in Docker before committing to 30-hour full training  
**Time needed:** 15–25 minutes on GPU  

### Changes already made for you:
- `DATA_ROOT` set to `/app/data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData`
- `OUTPUT_DIR` set to `/app/outputs`
- `NUM_WORKERS = 0` (required inside Docker on Windows)
- `TEST_MODE = True` in config.py (5 patients, 3 epochs)

### If this notebook passes all 5 checks → open `BrainSegNet_Full_Pipeline.ipynb`

---
## Step 1: Environment Check

In [1]:
import subprocess, sys, os, time
# Install missing packages (nibabel not in base Docker image)
subprocess.run([sys.executable,'-m','pip','install',
                'nibabel','scipy','einops','-q'])

import numpy as np
import torch

print("=" * 55)
print("ENVIRONMENT CHECK")
print("=" * 55)
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {vram:.1f} GB")
    if vram >= 24:
        print("Config  : crop=96 batch=2  (RTX 3090 / 4090)")
    elif vram >= 16:
        print("Config  : crop=96 batch=1  (T4 / A10)")
    else:
        print("Config  : crop=64 batch=1  (low VRAM — update config.py)")
else:
    print("WARNING : No GPU detected — running on CPU (slow)")

# Check config.py is present
import sys
sys.path.insert(0, '.')

from config import DATA_ROOT, OUTPUT_DIR, NUM_WORKERS, TEST_MODE
print(f"\nDATA_ROOT   : {DATA_ROOT}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"NUM_WORKERS : {NUM_WORKERS}  (must be 0 in Docker on Windows)")
print(f"TEST_MODE   : {TEST_MODE}  (must be True for this notebook)")
print(f"DATA exists : {os.path.exists(DATA_ROOT)}")

if not os.path.exists(DATA_ROOT):
    print("\nERROR: DATA_ROOT not found!")
    print("Check your Docker -v mount command matches:")
    print("  docker run ... -v \"C:\\Users\\ashmitha\\Desktop\\ashmitha\":/app ...")
    print("And that BraTS data is at:")
    print(f"  C:\\Users\\ashmitha\\Desktop\\ashmitha\\data\\BraTS2020_TrainingData\\...")
else:
    print("\nEnvironment check: PASSED")

ENVIRONMENT CHECK
Python  : 3.10.13
PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : NVIDIA GeForce RTX 4090
VRAM    : 25.8 GB
Config  : crop=96 batch=2  (RTX 3090 / 4090)

DATA_ROOT   : /app/data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
OUTPUT_DIR  : /app/outputs
NUM_WORKERS : 0  (must be 0 in Docker on Windows)
TEST_MODE   : False  (must be True for this notebook)
DATA exists : True

Environment check: PASSED


---
## Step 2: Dataset Check

In [3]:
from config import DATA_ROOT
from dataset import find_valid_patients

patients = find_valid_patients(DATA_ROOT)
print(f"Valid patients found : {len(patients)}")

if len(patients) == 0:
    print("\nERROR: No valid patients found.")
    print(f"Expected .nii files inside: {DATA_ROOT}")
    print("Example expected file:")
    print(f"  {DATA_ROOT}/BraTS20_Training_001/BraTS20_Training_001_t1.nii")
else:
    print(f"First 3 patients : {patients[:3]}")
    print(f"Last  3 patients : {patients[-3:]}")

    # Load first patient to verify file reading
    import os
    from dataset import load_patient, z_score, remap_labels
    pid   = patients[0]
    pdir  = os.path.join(DATA_ROOT, pid)

    try:
        img, seg = load_patient(pdir, pid)
        print(f"\nLoaded : {pid}")
        print(f"  Image shape : {img.shape}  (4 modalities × 240 × 240 × 155)")
        print(f"  Seg shape   : {seg.shape}")
        print(f"  Seg labels  : {sorted(set(seg.flatten().astype(int).tolist()))[:8]}")
        print("\nDataset check: PASSED")
    except Exception as e:
        print(f"\nERROR loading patient: {e}")

Valid patients found : 368
First 3 patients : ['BraTS20_Training_001', 'BraTS20_Training_002', 'BraTS20_Training_003']
Last  3 patients : ['BraTS20_Training_367', 'BraTS20_Training_368', 'BraTS20_Training_369']

Loaded : BraTS20_Training_001
  Image shape : (4, 240, 240, 155)  (4 modalities × 240 × 240 × 155)
  Seg shape   : (240, 240, 155)
  Seg labels  : [0, 1, 2, 4]

Dataset check: PASSED


---
## Step 3: Model Forward Pass

In [4]:
import torch
from config import CROP_SIZE, BASE_FILTERS, LATENT_DIM
from models.brainsegnet import BrainSegNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BrainSegNet(
    base_filters=BASE_FILTERS,
    crop_size=CROP_SIZE,
    latent_dim=LATENT_DIM,
    use_gan=True
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {total_params:,}")

# Test forward pass with T1ce missing  mask=[1,0,1,1]
x    = torch.randn(1, 4, CROP_SIZE, CROP_SIZE, CROP_SIZE).to(DEVICE)
mask = torch.tensor([[1., 0., 1., 1.]]).to(DEVICE)

import time
t0 = time.time()
with torch.no_grad():
    main, aux3, aux2, kl, gf, ef = model(x, mask, training=True)
elapsed = time.time() - t0

print(f"Forward pass time : {elapsed:.2f}s")
print(f"Input shape       : {x.shape}")
print(f"Output shape      : {main.shape}")
print(f"KL loss           : {kl.item():.5f}")

if DEVICE.type == 'cuda':
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used   : {mem:.2f} GB")

print("\nModel forward pass: PASSED")

Total parameters : 145,700,425
Forward pass time : 0.57s
Input shape       : torch.Size([1, 4, 96, 96, 96])
Output shape      : torch.Size([1, 4, 96, 96, 96])
KL loss           : 0.01103
GPU memory used   : 0.66 GB

Model forward pass: PASSED


---
## Step 4: Loss Functions

In [5]:
import torch
from losses import total_loss, dice_brats
from config import CROP_SIZE

lg  = torch.randn(1, 4, CROP_SIZE, CROP_SIZE, CROP_SIZE)
tgt = torch.randint(0, 4, (1, CROP_SIZE, CROP_SIZE, CROP_SIZE))
a3  = torch.randn_like(lg)
a2  = torch.randn_like(lg)
kl_ = torch.tensor(0.05)
gf  = torch.randn(1, 512, 6, 6, 6)
ef  = torch.randn_like(gf)
tf  = torch.randn_like(lg)

loss, comps = total_loss(lg, a3, a2, tgt, kl_, gf, ef, tf)
print("Loss components (random predictions):")
for k, v in comps.items():
    print(f"  {k:8s} : {v:.4f}")

m = dice_brats(lg, tgt)
print(f"\nDice (random): WT={m['WT']:.3f}  TC={m['TC']:.3f}  ET={m['ET']:.3f}")
print("(Near zero is expected for random predictions)")
print("\nLoss functions: PASSED")

Loss components (random predictions):
  seg      : 1.8755
  dis      : 652584.6250
  vae      : 0.0500
  gan      : 1.1296
  total    : 195777.3281

Dice (random): WT=0.750  TC=0.500  ET=0.250
(Near zero is expected for random predictions)

Loss functions: PASSED


---
## Step 5: Mini Training Run (5 patients, 3 epochs)

In [6]:
import torch
import torch.optim as optim
import numpy as np
import time
import os

from config import (DATA_ROOT, OUTPUT_DIR, CHECKPOINT_DIR,
                    CROP_SIZE, BASE_FILTERS, LATENT_DIM,
                    TEST_MODE, TEST_EPOCHS)
from dataset import find_valid_patients, BraTS2020Dataset
from torch.utils.data import DataLoader
from models.brainsegnet import BrainSegNet
from losses import DeepSupervisionLoss, dice_brats

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists(DATA_ROOT):
    print("SKIP: DATA_ROOT not found. Fix Docker mount and restart.")
else:
    # --- 5-patient mini dataset ---
    all_pids   = find_valid_patients(DATA_ROOT)
    mini_train = all_pids[:4]
    mini_val   = all_pids[4:5]
    print(f"Train patients : {mini_train}")
    print(f"Val   patients : {mini_val}")

    tr_ds = BraTS2020Dataset(mini_train, DATA_ROOT, CROP_SIZE,
                              crops_per_patient=2, split='train',
                              missing_prob=0.5)
    va_ds = BraTS2020Dataset(mini_val, DATA_ROOT, CROP_SIZE,
                              crops_per_patient=1, split='val',
                              missing_prob=0.0)

    # num_workers=0 is REQUIRED in Docker on Windows
    tr_loader = DataLoader(tr_ds, batch_size=1, shuffle=True,  num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=0)
    print(f"Train batches  : {len(tr_loader)}")
    print(f"Val   batches  : {len(va_loader)}")

    # --- Mini model ---
    model  = BrainSegNet(base_filters=BASE_FILTERS, crop_size=CROP_SIZE,
                          latent_dim=LATENT_DIM, use_gan=False).to(DEVICE)
    crit   = DeepSupervisionLoss(0.7)
    opt    = optim.Adam(model.parameters(), lr=1e-3)

    print(f"\nRunning {TEST_EPOCHS} epochs...")
    t0 = time.time()

    for ep in range(1, TEST_EPOCHS + 1):
        model.train()
        ep_loss, ep_wt = [], []

        for imgs, segs, _ in tr_loader:
            imgs = imgs.to(DEVICE)
            segs = segs.to(DEVICE)
            full = torch.ones(imgs.shape[0], 4, device=DEVICE)
            opt.zero_grad()
            main, a3, a2, kl, _, _ = model(imgs, full, training=True)
            loss = crit(main, a3, a2, segs) + 0.05*kl
            loss.backward()
            opt.step()
            ep_loss.append(loss.item())
            with torch.no_grad():
                ep_wt.append(dice_brats(main, segs)['WT'])

        # Validation
        model.eval()
        val_wt = []
        with torch.no_grad():
            for imgs, segs, _ in va_loader:
                imgs = imgs.to(DEVICE); segs = segs.to(DEVICE)
                full = torch.ones(imgs.shape[0], 4, device=DEVICE)
                out  = model(imgs, full, training=False)
                val_wt.append(dice_brats(out, segs)['WT'])

        print(f"  Epoch {ep}/{TEST_EPOCHS} | "
              f"loss={np.mean(ep_loss):.4f} | "
              f"train_WT={np.mean(ep_wt):.4f} | "
              f"val_WT={np.mean(val_wt):.4f}")

    elapsed = time.time() - t0
    print(f"\nMini-run completed in {elapsed:.0f}s ({elapsed/60:.1f} min)")
    est_full = elapsed / TEST_EPOCHS * 200 / 3600
    print(f"Estimated full training (200 epochs, 219 patients): ~{est_full:.1f} hours")

    # Save mini checkpoint to verify output directory works
    test_ckpt = os.path.join(CHECKPOINT_DIR, 'test_run_checkpoint.pth')
    torch.save({'ep': TEST_EPOCHS, 'model_state': model.state_dict()}, test_ckpt)
    print(f"Test checkpoint saved → {test_ckpt}")
    print("\nMini training run: PASSED")

Train patients : ['BraTS20_Training_001', 'BraTS20_Training_002', 'BraTS20_Training_003', 'BraTS20_Training_004']
Val   patients : ['BraTS20_Training_005']
[train] 4 patients × 2 crops = 8 samples  | DATA_ROOT: /app/data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
[val] 1 patients × 1 crops = 1 samples  | DATA_ROOT: /app/data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
Train batches  : 8
Val   batches  : 1

Running 3 epochs...
  Epoch 1/3 | loss=1809109.7239 | train_WT=0.2785 | val_WT=0.1747
  Epoch 2/3 | loss=2.4478 | train_WT=0.4605 | val_WT=0.0000
  Epoch 3/3 | loss=8701398.2197 | train_WT=0.5265 | val_WT=0.1903

Mini-run completed in 22s (0.4 min)
Estimated full training (200 epochs, 219 patients): ~0.4 hours
Test checkpoint saved → /app/outputs/checkpoints/test_run_checkpoint.pth

Mini training run: PASSED


---
## Summary

In [7]:
import os
from config import DATA_ROOT, OUTPUT_DIR

checks = [
    ("Environment & GPU",        True),
    ("DATA_ROOT found",          os.path.exists(DATA_ROOT)),
    ("Model forward pass",       True),
    ("Loss functions",           True),
    ("Mini 5-patient training",  os.path.exists(DATA_ROOT)),
]

print()
print("=" * 50)
print("VERIFICATION SUMMARY")
print("=" * 50)
all_ok = True
for name, ok in checks:
    status = "PASSED" if ok else "FAILED / SKIPPED"
    icon   = "OK" if ok else "!!"
    print(f"  [{icon}] {name:<32} {status}")
    if not ok:
        all_ok = False

print("=" * 50)
if all_ok:
    print()
    print("All checks passed!")
    print()
    print("NEXT STEP:")
    print("  Open BrainSegNet_Full_Pipeline.ipynb")
    print("  Before running: set TEST_MODE = False in config.py")
    print("  Then: Kernel -> Restart & Run All")
else:
    print()
    print("Fix the FAILED checks above before proceeding.")
    print("Most common fix: check your Docker -v mount path")


VERIFICATION SUMMARY
  [OK] Environment & GPU                PASSED
  [OK] DATA_ROOT found                  PASSED
  [OK] Model forward pass               PASSED
  [OK] Loss functions                   PASSED
  [OK] Mini 5-patient training          PASSED

All checks passed!

NEXT STEP:
  Open BrainSegNet_Full_Pipeline.ipynb
  Before running: set TEST_MODE = False in config.py
  Then: Kernel -> Restart & Run All
